# Partie 5 - Synthese Executive

Format court, redige pour un comite de direction. Ce notebook peut etre exporte en PDF 1 a 2 pages.


In [4]:
from pathlib import Path
import importlib
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import hr_analysis_utils as utils
importlib.reload(utils)

MahalanobisAnomalyDetector = utils.MahalanobisAnomalyDetector
random_oversample_minority = utils.random_oversample_minority
SimpleLogisticRegression = utils.SimpleLogisticRegression
attrition_by_group = utils.attrition_by_group
attrition_gap_summary = utils.attrition_gap_summary
coefficient_importance = utils.coefficient_importance
correlation_with_attrition = utils.correlation_with_attrition
dataset_overview = utils.dataset_overview
kpi_table = utils.kpi_table
load_data = utils.load_data
add_derived_columns = utils.add_derived_columns
missing_summary = utils.missing_summary
numeric_summary = utils.numeric_summary
plot_bar = utils.plot_bar
plot_box_by_attrition = utils.plot_box_by_attrition
plot_correlation_heatmap = utils.plot_correlation_heatmap
plot_histogram = utils.plot_histogram
plot_top_coefficients = utils.plot_top_coefficients
prepare_model_data = utils.prepare_model_data
prepare_numeric_anomaly_data = utils.prepare_numeric_anomaly_data
pr_auc_score_manual = utils.pr_auc_score_manual
roc_auc_score_manual = utils.roc_auc_score_manual
salary_gap_by_gender = utils.salary_gap_by_gender
average_by_group = utils.average_by_group
top_attrition_segments = utils.top_attrition_segments
find_best_threshold = utils.find_best_threshold
classification_metrics = utils.classification_metrics

DATA_PATH = PROJECT_ROOT / "raw" / "people_analytics_dataset.csv"
raw_df = pd.read_csv(DATA_PATH)
duplicate_employee_ids = int(raw_df["employee_id"].duplicated().sum()) if "employee_id" in raw_df.columns else 0
source_row_count = len(raw_df)
df = add_derived_columns(raw_df.drop_duplicates(subset=["employee_id"]).copy())
clean_row_count = len(df)
pd.set_option("display.max_columns", 100)


## 1. Ce qu'il faut retenir

- L'entreprise presente un **turnover eleve (18,77 %)**, avec des poches de risque bien identifiees.
- Les principaux signaux lies au depart sont une **securite psychologique plus faible**, un **absenteisme plus eleve**, davantage d'**heures supplementaires**, moins de **mobilite interne** et un **engagement plus bas**.
- Les zones a surveiller en priorite sont **l'Espagne**, **la France**, ainsi que les departements **HR**, **Finance** et **Sales**.


In [5]:
kpi_table(df).head(8)


,KPI,Valeur
0,Effectif total,8000.00
1,Taux d'attrition (%),18.77
2,Score d'engagement moyen,67.79
3,Score de securite psychologique moyen,53.72
4,Taux de promotion sur 3 ans (%),23.88
5,Taux de mobilite interne (%),61.91
6,Heures de formation moyennes,34.47
7,Absenteisme moyen (jours),13.88


## 2. Facteurs cles influencant turnover et engagement

### Turnover

- plus eleve dans certains segments organisationnels
- associe a des conditions d'emploi moins favorables dans ce dataset
- plus marque chez les collaborateurs les moins ancres dans l'organisation : moindre anciennete, moins de promotions et moins de mobilite interne
- desormais predible avec un niveau de performance utile pour la priorisation RH

### Engagement

- plus faible dans les populations qui quittent l'entreprise
- articule avec des leviers de management, de charge de travail, de reconnaissance et de securite psychologique


## 3. Resultat du modele predictif

Le modele detecte un **signal exploitable** :

- il est utile pour orienter la vigilance RH et prioriser les actions de retention
- il ne doit pas etre utilise seul pour prendre des decisions individuelles
- il confirme l'importance de la securite psychologique, de l'absenteisme, de l'engagement, de la mobilite interne et de la charge de travail dans la lecture du risque
- une experience de **detection d'anomalie** a egalement ete testee : elle reste moins robuste que le modele supervise


In [6]:
prepared = prepare_model_data(df, drop_columns=["employee_id"])
model = SimpleLogisticRegression(learning_rate=0.05, epochs=4000, reg_strength=0.02, class_weight="balanced")
model.fit(prepared.X_train, prepared.y_train)
best_threshold = find_best_threshold(prepared.y_val, model.predict_proba(prepared.X_val))
scores = model.predict_proba(prepared.X_test)

pd.DataFrame(
    {
        "Metrique": ["ROC-AUC", "PR-AUC", "Recall", "Precision"],
        "Valeur": [
            roc_auc_score_manual(prepared.y_test, scores),
            pr_auc_score_manual(prepared.y_test, scores),
            classification_metrics(prepared.y_test, scores, threshold=best_threshold["threshold"])["recall"],
            classification_metrics(prepared.y_test, scores, threshold=best_threshold["threshold"])["precision"],
        ],
    }
).round(4)


,Metrique,Valeur
0,ROC-AUC,0.8102
1,PR-AUC,0.5000
2,Recall,0.6844
3,Precision,0.4000


## 4. Recommandations operationnelles

1. Prioriser les plans d'action RH sur les segments les plus exposes au turnover.
2. Integrer l'engagement, la securite psychologique, l'absenteisme et la charge de travail dans un dispositif de veille RH trimestriel.
3. Renforcer les parcours de mobilite, de promotion et d'integration des profils les moins anciens comme leviers de retention.
4. Lancer une analyse plus fine de l'equite salariale et du role des variables sensibles.
5. Ameliorer la qualite des donnees, encadrer l'usage des variables sensibles et confirmer les resultats sur de nouvelles cohortes avant toute utilisation plus large.
